# Concrete Beam Section Calculation

This notebook provides a comprehensive analysis of concrete beam sections, including geometric properties, stress calculations, and reinforcement design based on engineering principles and design codes.

## 1. Import Required Libraries

Import necessary libraries for calculations and visualization.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import optimize
import warnings
warnings.filterwarnings('ignore')

# Set display options for better readability
pd.set_option('display.float_format', lambda x: f'{x:.4f}')
print("Libraries imported successfully!")

## 2. Define Beam Section Properties

Define the concrete beam dimensions, material properties, and load conditions.

In [ ]:
# Beam Section Dimensions (in mm)
b = 300  # Beam width
h = 600  # Beam height (total)
d = 540  # Effective depth (h - cover - bar diameter/2)
cover = 50  # Concrete cover to reinforcement

# Concrete Material Properties
f_c = 30  # Concrete compressive strength (MPa)
f_ct = 2.5  # Concrete tensile strength (MPa)
E_c = 5000 * np.sqrt(f_c)  # Young's modulus of concrete (MPa)

# Steel Material Properties
f_y = 500  # Steel yield strength (MPa)
E_s = 200000  # Young's modulus of steel (MPa)

# Load Conditions
M_applied = 150  # Applied bending moment (kN.m)
V_applied = 75   # Applied shear force (kN)

# Design parameters
steel_ratio = 0.015  # Assumed initial steel ratio
n_bars = 4  # Number of reinforcement bars
bar_diameter = 20  # Diameter of reinforcement bar (mm)
A_bar = np.pi * (bar_diameter / 2) ** 2  # Area of one bar (mm²)

# Display beam properties
print("=" * 60)
print("BEAM SECTION PROPERTIES")
print("=" * 60)
print(f"\nGeometric Properties:")
print(f"  Width (b): {b} mm")
print(f"  Height (h): {h} mm")
print(f"  Effective Depth (d): {d} mm")
print(f"  Concrete Cover: {cover} mm")

print(f"\nConcrete Properties:")
print(f"  Compressive Strength (f'_c): {f_c} MPa")
print(f"  Tensile Strength (f_ct): {f_ct} MPa")
print(f"  Modulus of Elasticity (E_c): {E_c:.0f} MPa")

print(f"\nSteel Properties:")
print(f"  Yield Strength (f_y): {f_y} MPa")
print(f"  Modulus of Elasticity (E_s): {E_s} MPa")

print(f"\nLoad Conditions:")
print(f"  Applied Bending Moment: {M_applied} kN.m")
print(f"  Applied Shear Force: {V_applied} kN")

print(f"\nReinforcement Properties:")
print(f"  Number of Bars: {n_bars}")
print(f"  Bar Diameter: {bar_diameter} mm")
print(f"  Area per Bar: {A_bar:.2f} mm²")
print("=" * 60)

## 3. Calculate Section Characteristics

Calculate moment of inertia, centroid location, and section modulus.

In [ ]:
# Calculate gross concrete section properties
A_c = b * h  # Gross concrete area (mm²)
I_c = (b * h ** 3) / 12  # Moment of inertia about centroid (mm⁴)
y_c = h / 2  # Distance from top to centroid (mm)

# Calculate reinforcement properties
A_s = n_bars * A_bar  # Total steel area (mm²)
rho = A_s / (b * d)  # Steel reinforcement ratio

# Effective depth from compression face
y_s = h - cover - (bar_diameter / 2)  # Depth to center of reinforcement

# Effective moment of inertia (accounting for concrete cracking - simplified)
# Using transformed section approach
n = E_s / E_c  # Modular ratio

# For uncracked section (elastic analysis)
# Centroid location considering both concrete and steel
A_total = A_c + (n - 1) * A_s
y_total = (A_c * y_c + (n - 1) * A_s * y_s) / A_total

# Moment of inertia for transformed section
I_transformed = I_c + A_c * (y_c - y_total) ** 2 + (n - 1) * A_s * (y_s - y_total) ** 2

# Section moduli
S_top = I_transformed / (y_total)  # Section modulus for top fiber
S_bot = I_transformed / (h - y_total)  # Section modulus for bottom fiber

print("=" * 60)
print("SECTION CHARACTERISTICS")
print("=" * 60)
print(f"\nGross Concrete Section:")
print(f"  Gross Area (A_c): {A_c:,.0f} mm²")
print(f"  Moment of Inertia (I_c): {I_c:.2e} mm⁴")
print(f"  Centroid from top (y_c): {y_c:.2f} mm")

print(f"\nReinforcement:")
print(f"  Total Steel Area (A_s): {A_s:.2f} mm²")
print(f"  Steel Ratio (ρ): {rho:.4f} ({rho*100:.2f}%)")
print(f"  Depth to centroid of steel (y_s): {y_s:.2f} mm")

print(f"\nTransformed Section (n = E_s/E_c = {n:.1f}):")
print(f"  Centroid from top (y): {y_total:.2f} mm")
print(f"  Moment of Inertia (I): {I_transformed:.2e} mm⁴")
print(f"  Section Modulus (top): {S_top:.2e} mm³")
print(f"  Section Modulus (bottom): {S_bot:.2e} mm³")
print("=" * 60)

## 4. Compute Bending Stress

Calculate bending stress using the formula: $\sigma = \frac{M \cdot y}{I}$

Where:
- M is the bending moment
- y is the distance from the neutral axis
- I is the moment of inertia

In [ ]:
# Convert applied moment to N.mm
M_Nmm = M_applied * 1e6  # kN.m to N.mm

# Calculate maximum bending stresses (elastic analysis)
# Stress at top fiber (compression)
stress_top = (M_Nmm * y_total) / I_transformed  # MPa

# Stress at bottom fiber (tension)
stress_bot = (M_Nmm * (h - y_total)) / I_transformed  # MPa

# Stress in concrete at reinforcement level
y_dist_steel = abs(y_s - y_total)
stress_concrete_at_steel = (M_Nmm * y_dist_steel) / I_transformed

# Stress in steel reinforcement (elastic)
stress_steel = n * stress_concrete_at_steel

# Shear stress (simplified average shear stress)
tau_avg = V_applied * 1e3 / (b * d)  # MPa

# Check flexural stresses
print("=" * 60)
print("BENDING STRESS ANALYSIS")
print("=" * 60)
print(f"\nApplied Moment: {M_applied} kN.m ({M_Nmm:.2e} N.mm)")
print(f"\nFlexural Stresses (Elastic Analysis):")
print(f"  Stress at top fiber (compression): {stress_top:.2f} MPa")
print(f"  Stress at bottom fiber (tension): {stress_bot:.2f} MPa")
print(f"  Stress in concrete at reinforcement: {stress_concrete_at_steel:.2f} MPa")
print(f"  Stress in steel reinforcement: {stress_steel:.2f} MPa")

print(f"\nShear Stress (average):")
print(f"  Applied Shear Force: {V_applied} kN")
print(f"  Average Shear Stress: {tau_avg:.2f} MPa")

print(f"\nCapacity Checks:")
print(f"  Concrete compression capacity: {0.4*f_c:.2f} MPa")
print(f"  Steel tension capacity: {f_y:.2f} MPa")
print(f"  Compression utilization: {(stress_top / (0.4*f_c))*100:.1f}%")
if stress_steel < f_y:
    print(f"  Steel tension utilization: {(stress_steel / f_y)*100:.1f}%")
else:
    print(f"  Steel tension utilization: {(stress_steel / f_y)*100:.1f}% - OVERSTRESS!")
print("=" * 60)

## 5. Determine Reinforcement Requirements

Calculate required steel reinforcement area using design codes, check stress limits, and compute reinforcement ratios.

In [ ]:
# Design factors (ACI/Eurocode approach)
phi = 0.9  # Resistance factor for flexure
gamma_c = 1.5  # Material factor for concrete
gamma_s = 1.15  # Material factor for steel

# Design moment (ultimate)
M_u = M_applied * 1.5  # Factored moment (1.5 load factor)

# Minimum and maximum reinforcement ratios
rho_min = max(0.33 / f_y, 0.002)  # Minimum reinforcement ratio
rho_max = 0.75 * (600 / (600 + f_y)) * (f_c / f_y)  # Maximum reinforcement ratio

# Balanced reinforcement ratio
rho_b = 0.85 * (f_c / f_y) * (600 / (600 + f_y))

# Required steel area using moment capacity equation
# For design: M_u = phi * A_s * f_y * (d - a/2)
# where a = A_s * f_y / (0.85 * f_c * b)
# Solving iteratively

def calculate_moment_capacity(A_s_test):
    """Calculate moment capacity for given steel area"""
    a = (A_s_test * f_y) / (0.85 * f_c * b)
    if a > d:
        return 0
    M_capacity = phi * A_s_test * f_y * (d - a / 2) / 1e6  # Convert to kN.m
    return M_capacity

# Find required steel area
from scipy.optimize import fsolve

def moment_deficit(A_s_test):
    return calculate_moment_capacity(A_s_test) - M_u

A_s_required = fsolve(moment_deficit, 1000)[0]
rho_required = A_s_required / (b * d)

# Neutral axis depth at balanced condition
a_b = 0.85 * (600 / (600 + f_y)) * d
c_b = a_b / 0.85

# Check current section
a_current = (A_s * f_y) / (0.85 * f_c * b)
M_capacity_current = calculate_moment_capacity(A_s)

print("=" * 60)
print("REINFORCEMENT DESIGN ANALYSIS")
print("=" * 60)
print(f"\nDesign Parameters:")
print(f"  Load Factor: 1.5")
print(f"  Resistance Factor (φ): {phi}")
print(f"  Factored Moment (M_u): {M_u:.2f} kN.m")

print(f"\nReinforcement Ratio Limits:")
print(f"  Minimum Ratio (ρ_min): {rho_min:.4f} ({rho_min*100:.2f}%)")
print(f"  Balanced Ratio (ρ_b): {rho_b:.4f} ({rho_b*100:.2f}%)")
print(f"  Maximum Ratio (ρ_max): {rho_max:.4f} ({rho_max*100:.2f}%)")
print(f"  Required Ratio (ρ_req): {rho_required:.4f} ({rho_required*100:.2f}%)")

print(f"\nRequired Reinforcement:")
print(f"  Required Steel Area: {A_s_required:.2f} mm²")
print(f"  Number of bars needed (Ø{bar_diameter}): {A_s_required/A_bar:.1f}")
print(f"  Current Steel Area: {A_s:.2f} mm²")
print(f"  Current Ratio (ρ): {rho:.4f} ({rho*100:.2f}%)")

print(f"\nCapacity Check:")
print(f"  Current Moment Capacity: {M_capacity_current:.2f} kN.m")
print(f"  Required Moment: {M_u:.2f} kN.m")
if M_capacity_current >= M_u:
    print(f"  Status: ✓ ADEQUATE (utilization: {(M_u/M_capacity_current)*100:.1f}%)")
else:
    print(f"  Status: ✗ INSUFFICIENT (additional area needed: {A_s_required - A_s:.2f} mm²)")

print(f"\nNeutral Axis:")
print(f"  Balanced depth (c_b): {c_b:.2f} mm")
print(f"  Current depth (c): {a_current/0.85:.2f} mm")
print("=" * 60)

## 6. Visualize Beam Section

Create visual diagrams of the beam cross-section showing dimensions, neutral axis, stress distribution, and reinforcement placement.

In [ ]:
# Create comprehensive visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# --- Subplot 1: Beam Section with Reinforcement ---
ax1 = axes[0, 0]
ax1.add_patch(plt.Rectangle((0, 0), b, h, fill=True, facecolor='lightgray', 
                             edgecolor='black', linewidth=2))

# Add reinforcement bars
bar_spacing = b / (n_bars + 1)
for i in range(1, n_bars + 1):
    x_bar = i * bar_spacing
    circle = plt.Circle((x_bar, y_s), bar_diameter/2, fill=True, 
                        facecolor='red', edgecolor='darkred', linewidth=1.5)
    ax1.add_patch(circle)

# Add neutral axis
ax1.axhline(y=y_total, color='blue', linestyle='--', linewidth=2, label='Neutral Axis')

# Add dimensions
ax1.arrow(0, -50, 0, -30, head_width=10, head_length=10, fc='black', ec='black')
ax1.arrow(b, -50, 0, -30, head_width=10, head_length=10, fc='black', ec='black')
ax1.text(b/2, -100, f'b = {b} mm', ha='center', fontsize=10, fontweight='bold')

ax1.arrow(-50, 0, 0, 0, head_width=10, head_length=10, fc='black', ec='black')
ax1.arrow(-50, h, 0, 0, head_width=10, head_length=10, fc='black', ec='black')
ax1.text(-120, h/2, f'h = {h} mm', ha='center', fontsize=10, fontweight='bold', rotation=90)

ax1.set_xlim(-200, b+100)
ax1.set_ylim(-150, h+100)
ax1.set_aspect('equal')
ax1.set_xlabel('Width (mm)', fontsize=10)
ax1.set_ylabel('Height (mm)', fontsize=10)
ax1.set_title('Beam Cross-Section with Reinforcement', fontsize=12, fontweight='bold')
ax1.legend(loc='upper right', fontsize=9)
ax1.grid(True, alpha=0.3)

# --- Subplot 2: Stress Distribution ---
ax2 = axes[0, 1]
y_values = np.linspace(0, h, 100)
stress_dist = (M_Nmm * (h - y_values)) / I_transformed

ax2.plot(stress_dist, y_values, 'b-', linewidth=2.5, label='Stress Distribution')
ax2.axhline(y=y_total, color='blue', linestyle='--', linewidth=2, label='Neutral Axis')
ax2.fill_betweenx(y_values, 0, stress_dist, alpha=0.3, color='blue')
ax2.axvline(x=0, color='black', linewidth=0.5)
ax2.set_xlabel('Bending Stress (MPa)', fontsize=10)
ax2.set_ylabel('Distance from Top (mm)', fontsize=10)
ax2.set_title('Flexural Stress Distribution', fontsize=12, fontweight='bold')
ax2.legend(loc='best', fontsize=9)
ax2.grid(True, alpha=0.3)

# --- Subplot 3: Properties Summary Table ---
ax3 = axes[1, 0]
ax3.axis('tight')
ax3.axis('off')

properties_data = [
    ['Property', 'Value', 'Unit'],
    ['Width (b)', f'{b:.0f}', 'mm'],
    ['Height (h)', f'{h:.0f}', 'mm'],
    ['Effective Depth (d)', f'{d:.0f}', 'mm'],
    ['Concrete Strength (f\'c)', f'{f_c:.0f}', 'MPa'],
    ['Steel Strength (fy)', f'{f_y:.0f}', 'MPa'],
    ['Steel Area (As)', f'{A_s:.2f}', 'mm²'],
    ['Steel Ratio (ρ)', f'{rho*100:.2f}', '%'],
    ['Moment of Inertia (I)', f'{I_transformed:.2e}', 'mm⁴'],
]

table = ax3.table(cellText=properties_data, cellLoc='center', loc='center',
                  colWidths=[0.4, 0.3, 0.3])
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 2)

# Style header row
for i in range(3):
    table[(0, i)].set_facecolor('#40466e')
    table[(0, i)].set_text_props(weight='bold', color='white')

# Alternate row colors
for i in range(1, len(properties_data)):
    for j in range(3):
        if i % 2 == 0:
            table[(i, j)].set_facecolor('#f0f0f0')
        else:
            table[(i, j)].set_facecolor('#ffffff')

ax3.set_title('Section Properties Summary', fontsize=12, fontweight='bold', pad=20)

# --- Subplot 4: Capacity Analysis ---
ax4 = axes[1, 1]
ax4.axis('tight')
ax4.axis('off')

capacity_data = [
    ['Parameter', 'Value', 'Limit/Unit'],
    ['Applied Moment', f'{M_applied:.1f}', 'kN.m'],
    ['Moment Capacity', f'{M_capacity_current:.1f}', 'kN.m'],
    ['Utilization', f'{(M_applied/M_capacity_current)*100:.1f}', '%'],
    ['Concrete Stress (top)', f'{stress_top:.2f}', 'MPa'],
    ['Concrete Allowable', f'{0.4*f_c:.2f}', 'MPa'],
    ['Steel Stress', f'{stress_steel:.2f}', 'MPa'],
    ['Steel Allowable', f'{f_y:.2f}', 'MPa'],
    ['Steel Ratio (current)', f'{rho*100:.2f}', '%'],
    ['Steel Ratio (min)', f'{rho_min*100:.2f}', '%'],
]

table2 = ax4.table(cellText=capacity_data, cellLoc='center', loc='center',
                   colWidths=[0.4, 0.3, 0.3])
table2.auto_set_font_size(False)
table2.set_fontsize(9)
table2.scale(1, 2)

# Style header row
for i in range(3):
    table2[(0, i)].set_facecolor('#40466e')
    table2[(0, i)].set_text_props(weight='bold', color='white')

# Alternate row colors
for i in range(1, len(capacity_data)):
    for j in range(3):
        if i % 2 == 0:
            table2[(i, j)].set_facecolor('#f0f0f0')
        else:
            table2[(i, j)].set_facecolor('#ffffff')

ax4.set_title('Capacity & Stress Analysis', fontsize=12, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig('beam_section_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Visualization complete!")

## Summary

This notebook provides a complete framework for analyzing concrete beam sections, including:

- **Geometric Analysis**: Calculation of cross-sectional properties (area, moment of inertia, centroid)
- **Elastic Stress Analysis**: Computation of flexural and shear stresses under applied loads
- **Reinforcement Design**: Determination of required steel area based on design codes
- **Capacity Verification**: Check of section capacity against design moments and stresses
- **Visual Representation**: Comprehensive diagrams showing section geometry, stress distribution, and reinforcement layout

The analysis can be easily modified by adjusting the section dimensions, material properties, and applied loads in Section 2. All calculations follow standard concrete design principles compatible with ACI and Eurocode approaches.